In [102]:
# IMPORTS
# FIRST, pip install -r requirements.txt

import json
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

In [ ]:
# PATHS
ROOT = Path(".")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results" / "playlists"

for d in (DATA_PROC, MODELS_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

USER_FILES = {
    "andy": DATA_RAW / "enrichment-success-andy.json",
    "dishita": DATA_RAW / "enrichment-success-dish.json",
    "riya": DATA_RAW / "enrichment-success-riya.json",
    "priyanka": DATA_RAW / "enrichment-success-priyanka.json",
}

USERS = list(USER_FILES.keys())

# TUNEABLE PARAMETERS FOR ENTIRE PIPELINE
AUDIO_FEATURES = [
    "danceability", "energy", "valence", "tempo", "loudness",
    "acousticness", "instrumentalness", "speechiness", "liveness",
    "key", "mode", "time_signature",
]
MOOD_FEATURES = ["valence", "energy", "danceability", "tempo"]

# DATA LOADING & PREPROCESSING | TRACK TABLES
MIN_NEIGHBOR_MS = 30_000
TFIDF_MIN_DF = 2
TFIDF_MAX_FEATURES = 500

# USER PROFILES
MIN_SKIP_RATE_DISLIKE = 0.7
MAX_MS_DISLIKE = 15_000
N_MOOD_CLUSTERS = 4

In [104]:
# DATA LOADING AND PREPROCESSING

"""
Returns the best available tag list for a track.
 - Tries all_tags first.
 - If empty or missing, falls back to sc_genres.
 - If still empty, falls back to lastfm_tags.
 - Returns a list of lowercase, stripped strings.
 - If none are available, returns an empty list.
"""
def resolve_tags_columns(df: pd.DataFrame) -> pd.Series:
    tags = df["all_tags"]

    mask = tags.isna() | (tags.str.len() == 0)
    tags = tags.where(~mask, df["sc_genres"])

    mask = tags.isna() | (tags.str.len() == 0)
    tags = tags.where(~mask, df["lastfm_tags"])

    tags = tags.apply(
        lambda t: [x.strip().lower() for x in t] if isinstance(t, list) else []
    )

    return tags

"""
Loads a single user's raw (enriched) JSON listening history and returns it as a cleaned df.
 - Filters out non-track plays (episodes, audiobooks)
 - Renames audio feature columns to simplify access/keep consistency
 - Converts timestamps to pandas datetime
 - Ensures numeric values for ms_played
 - Marks skipped tracks as boolean
 - Adds 'tags' column
 - Adds 'meaningful' column
"""
def load_user_events(path: Path, user: str) -> pd.DataFrame:
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)

    df = pd.json_normalize(raw)
    df['user'] = user

    df.rename(columns={f"audio_features.{f}": f for f in AUDIO_FEATURES}, inplace=True)
    mask = df["spotify_track_uri"].notna() & df["episode_name"].isna()
    df = df.loc[mask]

    df["ts"] = pd.to_datetime(df["ts"], utc=True)
    df["ms_played"] = pd.to_numeric(df["ms_played"], errors="coerce").fillna(0)
    df["skipped"] = df["skipped"].fillna(False).astype(bool)
    df["tags"] = resolve_tags_columns(df)

    # Meaningful play: not skipped AND listened to >= 30 seconds
    df["meaningful"] = (~df["skipped"]) & (df["ms_played"] >= MIN_MEANINGFUL_MS)
    return df

"""
Loads and combines listening history for all users into a single df.
Each row represents one play event, tagged with the user it belongs to.
"""
def load_all_events(user_files: dict) -> pd.DataFrame:
    frames = []

    for user, path in user_files.items():
        df = load_user_events(path, user)
        frames.append(df)

    return pd.concat(frames, ignore_index=True)

events = load_all_events(USER_FILES)

# Save raw events for reference
events.to_csv(DATA_PROC / "events.csv", index=False)

print(f"Total play events: {len(events):,}")
print(f"Unique tracks: {events['spotify_track_uri'].nunique():,}")
print(f"Users: {events['user'].unique()}")

Total play events: 29,887
Unique tracks: 7,068
Users: <StringArray>
['andy', 'dishita', 'riya', 'priyanka']
Length: 4, dtype: str


In [105]:
# TRACK TABLE (ITEM VECTORS)

# COLUMNS TO INCLUDE IN THE TRACK TABLE
cols = [
    "spotify_track_uri",
    "master_metadata_track_name",
    "master_metadata_album_artist_name",
    "master_metadata_album_album_name",
    "tags",
    *AUDIO_FEATURES,
]

# BUILD TRACK TABLE: ONE ROW PER UNQIUE TRACK URI
track_table = (
    events[cols]
    .drop_duplicates("spotify_track_uri")
    .rename(columns={
        "master_metadata_track_name": "track_name",
        "master_metadata_album_artist_name": "artist_name",
        "master_metadata_album_album_name": "album_name",
    })
    .reset_index(drop=True)
)

# ASSIGN A UNIQUE INTEGER ID TO EACH TRACK
track_table["track_id"] = np.arange(len(track_table))

# LOOKUP DICTIONARIES (FOR FASTER MAPPING)
uri_to_id = track_table.set_index("spotify_track_uri")["track_id"].to_dict()
id_to_uri = track_table.set_index("track_id")["spotify_track_uri"].to_dict()

# NORMALIZED AUDIO FEATURE MATRIX
audio_matrix = track_table[AUDIO_FEATURES].fillna(0).to_numpy(dtype=float)
audio_scaler = MinMaxScaler()
audio_matrix_norm = audio_scaler.fit_transform(audio_matrix)

# TAG TF-IDF MATRIX
tag_docs = track_table["tags"].str.join(" ").fillna("")
tfidf = TfidfVectorizer(min_df=TFIDF_MIN_DF, max_features=TFIDF_MAX_FEATURES)
tag_matrix = tfidf.fit_transform(tag_docs)
tag_vocab = tfidf.get_feature_names_out()

# CONVERT ARTIST NAME TO ARTIST ID
track_table["artist_id"] = pd.Categorical(track_table["artist_name"]).codes

# SAVE TRACK TABLE & FITTED MODELS
track_table.to_csv(DATA_PROC / "track_features.csv", index=False)
joblib.dump(audio_scaler, MODELS_DIR / "audio_scaler.joblib")
joblib.dump(tfidf, MODELS_DIR / "tfidf.joblib")

print(f"Track table saved: {len(track_table):,} tracks, tag vocab = {len(tag_vocab)}")
print(f"Tag vocab size: {len(tag_vocab)}")

Track table saved: 7,068 tracks, tag vocab = 330
Tag vocab size: 330


In [106]:
# USER PROFILES

# MAP TRACK URIS TO IDS
# Each track URI gets a unique integer ID for indexing into matrices.
events["track_id"] = events["spotify_track_uri"].map(uri_to_id)

# PER-(USER, TRACK) PLAY-STATS
"""
Aggregates raw play events to compute per-user, per-track statistics:
- n_plays: total plays of the track
- avg_ms: average milliseconds listened
- skip_count: number of skipped plays
- meaningful_pct: fraction of plays considered meaningful (>=30s, not skipped)
"""
play_stats = (
    events.groupby(["user", "track_id"])
    .agg(
        n_plays = ("ms_played", "count"),
        avg_ms = ("ms_played", "mean"),
        skip_count = ("skipped", "sum"),
        meaningful_pct = ("meaningful", "mean"),
    )
    .reset_index()
)

# Fraction of plays skipped for each user-track
play_stats["skip_rate"] = play_stats["skip_count"] / play_stats["n_plays"]

# PLAY WEIGHT (VECTORIZED)
"""
Assigns a weight to a (user, track) play record showing how much the user likes the track.
 - Tracks that are frequently skipped early get a weight of 0.
 - Otherwise, weight increases with the percentage of meaningful listens and 
    the log of total play count (to dampen the reduced impact of excessive replays).
"""
play_stats["weight"] = (play_stats["meaningful_pct"] * np.log1p(play_stats["n_plays"]))

dislike_mask = (
    (play_stats["skip_rate"] > MIN_SKIP_RATE_DISLIKE)
    & (play_stats["avg_ms"] < MAX_MS_DISLIKE)
)

play_stats.loc[dislike_mask, "weight"] = 0.0
play_stats.to_csv(DATA_PROC / "play_stats.csv", index=False)

# Map users to row indices for the sparse matrix
user_to_idx = {u: i for i, u in enumerate(USERS)}
play_stats["user_idx"] = play_stats["user"].map(user_to_idx)

# Build a sparse user × track weight matrix
# Rows = users, Columns = tracks, Values = play weight
user_track_matrix = csr_matrix(
    (play_stats["weight"].values, (play_stats["user_idx"].values, play_stats["track_id"].values)),
    shape=(len(USERS), audio_matrix_norm.shape[0])
)

# Precompute sum of weights per user to normalize later
weight_sum = np.array(user_track_matrix.sum(axis=1)).flatten()

# LONG-TERM AUDIO PROFILES
"""
Builds a single vector representing each user's overall audio taste.
- Weighted average of all tracks the user listened to.
- Tracks with higher engagement contribute more to the profile.
"""
long_term_profiles = user_track_matrix @ audio_matrix_norm
long_term_profiles = long_term_profiles / (weight_sum[:, None] + 1e-9)

# TAG DISTRIBUTIONS PER USER
"""
Builds weighted tag profiles for all users in one pass.
- Captures which genres and descriptors are characteristic of each user's taste.
- Weighted by play engagement.
"""
user_tag_profiles = user_track_matrix @ tag_matrix
user_tag_profiles = user_tag_profiles / (weight_sum[:, None] + 1e-9)

# MOOD CLUSTERS PER USER
"""
Identifies mood-based clusters in a user's listening history.
- Uses a subset of audio features (MOOD_FEATURES).
- Weighted KMeans clustering per user.
- Returns cluster centroids, labels, track indices, and weighted tag distributions per cluster.
"""
# Extract mood features and fill NaNs with 0 before scaling
mood_feat_idx = [AUDIO_FEATURES.index(f) for f in MOOD_FEATURES]
mood_matrix = audio_matrix[:, mood_feat_idx].copy()

# Fill NaNs with 0
mood_matrix = np.nan_to_num(mood_matrix, nan=0.0)

# Scale to [0,1]
mood_scaler = MinMaxScaler()
mood_matrix_norm = mood_scaler.fit_transform(mood_matrix)

def build_mood_clusters(user: str):
    u_idx = user_to_idx[user]
    idx = user_track_matrix[u_idx].indices
    w = user_track_matrix[u_idx].data

    # Handle users with no meaningful plays
    if len(idx) == 0:
        return np.empty((0, len(MOOD_FEATURES))), np.array([]), np.array([]), {}

    X = mood_matrix_norm[idx]
    k = min(N_MOOD_CLUSTERS, len(idx))
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X, sample_weight=w)

    tag_dist = {}
    for c in range(k):
        mask = labels == c
        c_idx, c_w = idx[mask], w[mask]
        if len(c_idx) == 0:
            tag_dist[c] = np.zeros(tag_matrix.shape[1])
        else:
            ws = np.array(tag_matrix[c_idx].multiply(c_w[:, None]).sum(axis=0)).flatten()
            tag_dist[c] = ws / (c_w.sum() + 1e-9)
    return km.cluster_centers_, labels, idx, tag_dist

print("Building mood clusters...")
mood_clusters = {u: build_mood_clusters(u) for u in USERS}

# SAVE LONG-TERM AUDIO PROFILES
# Each row = user
# Each column = audio feature
pd.DataFrame(long_term_profiles, columns=AUDIO_FEATURES) \
    .assign(user=USERS) \
    .to_csv(DATA_PROC / "user_profiles.csv", index=False)

# SAVE TAG PROFILES
# Ensure dense array for pandas (TF-IDF matrix is sparse).
def to_dense(matrix):
    # Convert sparse matrix to dense NumPy array if needed.
    return matrix.toarray() if hasattr(matrix, "toarray") else matrix

user_tag_profiles_dense = to_dense(user_tag_profiles)

# Each row = user
# Each column = tag
pd.DataFrame(user_tag_profiles_dense, columns=tag_vocab) \
    .assign(user=USERS) \
    .to_csv(DATA_PROC / "user_tag_profiles.csv", index=False)

# SAVE MOOD CLUSTERS PER USER
# One CSV per user
# Rows = cluster centroids
for u in USERS:
    centroids, *_ = mood_clusters[u]
    pd.DataFrame(centroids, columns=MOOD_FEATURES) \
        .rename_axis("cluster") \
        .reset_index() \
        .assign(user=u) \
        .to_csv(DATA_PROC / f"mood_clusters_{u}.csv", index=False)
    
# Save fitted mood scaler for later use
joblib.dump(mood_scaler, MODELS_DIR / "mood_scaler.joblib")

# USER-USER SIMILARITY
"""
Computes pairwise user similarity using the tag profiles.
Uses cosine similarity between all users.
"""
user_sim_df = pd.DataFrame(
    cosine_similarity(user_tag_profiles),
    index=USERS, columns=USERS
)

print("\nUser profiles saved -> data/processed/")
print("mood_scaler.joblib -> models/")
print("\n", user_sim_df.round(3))


Building mood clusters...

User profiles saved -> data/processed/
mood_scaler.joblib -> models/

            andy  dishita   riya  priyanka
andy      1.000    0.835  0.534     0.906
dishita   0.835    1.000  0.628     0.818
riya      0.534    0.628  1.000     0.741
priyanka  0.906    0.818  0.741     1.000


In [ ]:
# CANDIDATE GENERATION

In [ ]:
# SCORING

# RE-RANKING

# UI-WIDGETS (?)

# TESTING

# DEMO (?)